In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Milestone 1**

In [3]:


import os
import glob
import random
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm


DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
SR = 22050
DURATION = 5.0
TOP_DB = 20

random.seed(67)
np.random.seed(67)


GENRES = sorted([g for g in os.listdir(DATA_ROOT) 
                 if os.path.isdir(os.path.join(DATA_ROOT, g))])

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
STEMS = {
    'drums.wav': 'drums',
    'vocals.wav': 'vocals',
    'bass.wav': 'bass',
    'other.wav': 'other'
}


In [4]:


def build_dataset(root_dir, val_split=0.17, seed=42):

    train_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {k: [] for k in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)

    corrupted_count = 0
    less_5_0491MB = 0
    greater_5_0493MB = 0

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        songs = sorted(os.listdir(genre_path))

        valid_songs = []

        for song in songs:
            song_path = os.path.join(genre_path, song)
            stem_files = []

            for stem_file in STEMS:
                fpath = os.path.join(song_path, stem_file)
                if not os.path.exists(fpath):
                    break
                size = os.path.getsize(fpath)

                if size < 4 * 1024:
                    corrupted_count += 1

                if size < 5.0491 * 1024 * 1024:
                    less_5_0491MB += 1

                if size > 5.0493 * 1024 * 1024:
                    greater_5_0493MB += 1

                stem_files.append(fpath)

            if len(stem_files) == 4:
                valid_songs.append(song_path)

        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        for s in train_songs:
            for stem_file in STEMS:
                train_dataset[genre][STEMS[stem_file]].append(
                    os.path.join(s, stem_file)
                )

        for s in val_songs:
            for stem_file in STEMS:
                val_dataset[genre][STEMS[stem_file]].append(
                    os.path.join(s, stem_file)
                )

    print("\n--- Q1 ---")
    print("Corrupted + (<5.0491MB):", corrupted_count + less_5_0491MB)

    print("\n--- Q2 ---")
    print("Absolute difference:",
          abs(greater_5_0493MB - less_5_0491MB))

    print("\n--- Q3 ---")
    reggae_train_drums = len(train_dataset['reggae']['drums'])
    country_val_vocals = len(val_dataset['country']['vocals'])
    print("Absolute difference:",
          abs(reggae_train_drums - country_val_vocals))

    return train_dataset, val_dataset


tr, val = build_dataset(DATA_ROOT)



--- Q1 ---
Corrupted + (<5.0491MB): 1256

--- Q2 ---
Absolute difference: 1072

--- Q3 ---
Absolute difference: 66


In [5]:


def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):

    records = []

    for genre in dataset_dict:
        for stem in dataset_dict[genre]:
            for file_path in tqdm(dataset_dict[genre][stem], leave=False):

                y, _ = librosa.load(file_path, sr=sr)
                total_duration = len(y) / sr

                intervals = librosa.effects.split(y, top_db=top_db)

                silence_type = []
                max_silence = 0

                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("Full")

                else:
                    if intervals[0][0] > 0:
                        start_silence = intervals[0][0] / sr
                        max_silence = max(max_silence, start_silence)
                        silence_type.append("Start")

                    if intervals[-1][1] < len(y):
                        end_silence = (len(y) - intervals[-1][1]) / sr
                        max_silence = max(max_silence, end_silence)
                        silence_type.append("End")

                    for i in range(len(intervals)-1):
                        gap = (intervals[i+1][0] - intervals[i][1]) / sr
                        if gap > 0:
                            max_silence = max(max_silence, gap)
                            silence_type.append("Middle")

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem,
                        "Duration": round(total_duration,2),
                        "Max_Silence_Sec": round(max_silence,2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)
    return df


In [6]:
# ===============================
# CELL 4: Q4–Q9 RESULTS
# ===============================

df_silence = find_long_silences(tr)

print("\n--- Q4 ---")
print("Total files silence >=5:", len(df_silence))

print("\n--- Q5 ---")
print("Vocals silence >=5:",
      len(df_silence[df_silence['Stem']=='vocals']))

print("\n--- Q6 ---")
print("Average silence vocals:",
      df_silence[df_silence['Stem']=='vocals']['Max_Silence_Sec'].mean())

print("\n--- Q7 ---")
print("Jazz drums silence >=5:",
      len(df_silence[(df_silence['Genre']=='jazz') &
                     (df_silence['Stem']=='drums')]))

print("\n--- Q8 ---")
print("Jazz drums middle only:",
      len(df_silence[(df_silence['Genre']=='jazz') &
                     (df_silence['Stem']=='drums') &
                     (df_silence['Silence_Location']=='Middle')]))

print("\n--- Q9 ---")
print("Jazz drums silence >=10:",
      len(df_silence[(df_silence['Genre']=='jazz') &
                     (df_silence['Stem']=='drums') &
                     (df_silence['Max_Silence_Sec']>=10)]))



--- Q4 ---
Total files silence >=5: 680

--- Q5 ---
Vocals silence >=5: 304

--- Q6 ---
Average silence vocals: 12.590789473684211

--- Q7 ---
Jazz drums silence >=5: 24

--- Q8 ---
Jazz drums middle only: 0

--- Q9 ---
Jazz drums silence >=10: 7


In [7]:
# ===============================
# CELL 5: Q10–Q12 MIX SAMPLE
# ===============================

rock_songs = sorted(os.listdir(os.path.join(DATA_ROOT,'rock')))
first_song = rock_songs[0]

stems_audio = []
for stem_file in STEMS:
    path = os.path.join(DATA_ROOT,'rock',first_song,stem_file)
    y,_ = librosa.load(path, sr=SR, duration=5.0)
    stems_audio.append(y)

stems_stack = np.vstack(stems_audio)
mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw**2))
max_val = np.max(np.abs(mix_raw))

print("\n--- Q10 ---")
print("Mix length:", len(mix_raw))

print("\n--- Q11 ---")
print("RMS:", round(rms_val,2))

print("\n--- Q12 ---")
print("Max peak before norm:", max_val)



--- Q10 ---
Mix length: 110250

--- Q11 ---
RMS: 0.2

--- Q12 ---
Max peak before norm: 0.96006984


In [ ]:
!pip install --upgrade --force-reinstall wandb --quiet


In [ ]:
!pip install wandb==0.15.9 --quiet


In [ ]:
import wandb

# Direct login (no key needed if already logged in)
wandb.login()

run = wandb.init(
    project="23f3000162-t12026",
    name="CNN_Scratch_V1",
    settings=wandb.Settings(console="off")  # important for Kaggle
)

print("initialized successfully")


In [ ]:
# import random
# import wandb

# for epoch in range(5):
#     loss = random.uniform(0.5, 1.5)
#     f1 = random.uniform(0.6, 0.9)
    
#     wandb.log({
#         "train_loss": loss,
#         "macro_f1": f1
#     })


In [ ]:
# # ========== CELL 3 : Basic Dataset Statistics ==========

# import os

# BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# GENRES_PATH = os.path.join(BASE_PATH, "genres_stems")

# genres = sorted(os.listdir(GENRES_PATH))

# print("Total Genres:", len(genres))
# print("-" * 40)

# for genre in genres:
#     genre_path = os.path.join(GENRES_PATH, genre)
#     songs = os.listdir(genre_path)
#     print(f"{genre}: {len(songs)} songs")


In [ ]:
# # ========== CELL 4 : Audio Length Distribution ==========

# import librosa
# import random
# import numpy as np

# lengths = []

# # Har genre se 3 random songs check karenge (fast EDA)
# for genre in genres:
#     genre_path = os.path.join(GENRES_PATH, genre)
#     songs = os.listdir(genre_path)
    
#     sample_songs = random.sample(songs, 3)
    
#     for song in sample_songs:
#         stem_path = os.path.join(genre_path, song, "drums.wav")
#         y, sr = librosa.load(stem_path, sr=None)
#         duration = len(y) / sr
#         lengths.append(duration)

# print("Min Duration:", round(min(lengths), 2), "seconds")
# print("Max Duration:", round(max(lengths), 2), "seconds")
# print("Average Duration:", round(np.mean(lengths), 2), "seconds")


In [ ]:
# # ========== CELL 5 : Waveform + Spectrogram Visualization ==========

# import matplotlib.pyplot as plt
# import librosa.display

# # Ek random sample lo
# sample_genre = random.choice(genres)
# genre_path = os.path.join(GENRES_PATH, sample_genre)
# sample_song = random.choice(os.listdir(genre_path))

# stem_path = os.path.join(genre_path, sample_song, "drums.wav")

# y, sr = librosa.load(stem_path, sr=22050)

# # Waveform
# plt.figure(figsize=(12, 4))
# librosa.display.waveshow(y, sr=sr)
# plt.title(f"Waveform - {sample_genre}")
# plt.show()

# # Spectrogram
# mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
# mel_db = librosa.power_to_db(mel)

# plt.figure(figsize=(12, 4))
# librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel')
# plt.title("Log-Mel Spectrogram")
# plt.colorbar()
# plt.show()


In [ ]:
# # ========== CELL 6 : Audio Preprocessing Functions ==========

# import numpy as np
# import librosa
# import random

# TARGET_SR = 22050
# TARGET_DURATION = 10  # seconds
# TARGET_LENGTH = TARGET_SR * TARGET_DURATION

# def load_audio(path):
#     y, sr = librosa.load(path, sr=TARGET_SR)
#     return y

# def random_crop(y):
#     if len(y) > TARGET_LENGTH:
#         start = random.randint(0, len(y) - TARGET_LENGTH)
#         y = y[start:start + TARGET_LENGTH]
#     else:
#         y = np.pad(y, (0, TARGET_LENGTH - len(y)))
#     return y

# def normalize_audio(y):
#     return y / (np.max(np.abs(y)) + 1e-6)

# def extract_log_mel(y):
#     mel = librosa.feature.melspectrogram(
#         y=y,
#         sr=TARGET_SR,
#         n_mels=128,
#         hop_length=512
#     )
#     mel_db = librosa.power_to_db(mel)
#     return mel_db


In [ ]:
# # ========== CELL 7 : Mashup + Noise Augmentation ==========

# ESC_PATH = os.path.join(BASE_PATH, "ESC-50-master/audio")
# noise_files = os.listdir(ESC_PATH)

# def create_mashup(song_folder):
#     stems = ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]
#     audios = []
    
#     for stem in stems:
#         path = os.path.join(song_folder, stem)
#         y = load_audio(path)
#         y = random_crop(y)
#         y = normalize_audio(y)
        
#         # Random volume scaling (instrument imbalance simulate)
#         scale = random.uniform(0.6, 1.4)
#         y = y * scale
        
#         audios.append(y)
    
#     mixed = np.sum(audios, axis=0)
#     return normalize_audio(mixed)

# def add_noise(y):
#     noise_file = random.choice(noise_files)
#     noise_path = os.path.join(ESC_PATH, noise_file)
    
#     noise = load_audio(noise_path)
#     noise = random_crop(noise)
#     noise = normalize_audio(noise)
    
#     noise_level = random.uniform(0.01, 0.05)
#     y = y + noise_level * noise
    
#     return normalize_audio(y)


In [ ]:
# # ========== SAFE MASHUP FUNCTION ==========

# def create_mashup(song_folder):
#     stems = ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]
#     audios = []
    
#     for stem in stems:
#         path = os.path.join(song_folder, stem)
        
#         # Skip if stem missing
#         if not os.path.exists(path):
#             return None
        
#         y = load_audio(path)
#         y = random_crop(y)
#         y = normalize_audio(y)
        
#         scale = random.uniform(0.6, 1.4)
#         y = y * scale
        
#         audios.append(y)
    
#     mixed = np.sum(audios, axis=0)
#     return normalize_audio(mixed)


In [ ]:
# # ========== SAFE MASHUP FUNCTION ==========

# def create_mashup(song_folder):
#     stems = ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]
#     audios = []
    
#     for stem in stems:
#         path = os.path.join(song_folder, stem)
        
#         # Skip if stem missing
#         if not os.path.exists(path):
#             return None
        
#         y = load_audio(path)
#         y = random_crop(y)
#         y = normalize_audio(y)
        
#         scale = random.uniform(0.6, 1.4)
#         y = y * scale
        
#         audios.append(y)
    
#     mixed = np.sum(audios, axis=0)
#     return normalize_audio(mixed)


In [ ]:
# # ========== Re-define Dataset Paths & Genres ==========

# import os

# BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# GENRES_PATH = os.path.join(BASE_PATH, "genres_stems")

# genres = sorted(os.listdir(GENRES_PATH))

# print("Genres loaded:", genres)


In [ ]:
# import torch
# from torch.utils.data import Dataset

# label_map = {genre: idx for idx, genre in enumerate(genres)}

# class MashupDataset(Dataset):
#     def __init__(self, folders):
#         self.folders = folders
        
#     def __len__(self):
#         return len(self.folders)
    
#     def __getitem__(self, idx):
#         folder = self.folders[idx]
        
#         genre = folder.split("/")[-2]
#         label = label_map[genre]
        
#         y = create_mashup(folder)
#         y = add_noise(y)
        
#         mel = extract_log_mel(y)
        
#         mel = torch.tensor(mel).unsqueeze(0).float()
#         label = torch.tensor(label).long()
        
#         return mel, label


In [ ]:
# # ========== DEBUG STRUCTURE CHECK ==========

# for genre in genres:
#     genre_path = os.path.join(GENRES_PATH, genre)
#     songs = os.listdir(genre_path)
    
#     print("Example genre:", genre)
#     print("First 3 folders:", songs[:3])
    
#     sample_folder = os.path.join(genre_path, songs[0])
#     print("Files inside first folder:", os.listdir(sample_folder))
#     break


In [ ]:
# # ========== Correct valid_folders Creation ==========

# valid_folders = []

# stems = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

# for genre in genres:
#     genre_path = os.path.join(GENRES_PATH, genre)
#     songs = os.listdir(genre_path)
    
#     for song in songs:
#         song_folder = os.path.join(genre_path, song)
        
#         if all(os.path.exists(os.path.join(song_folder, s)) for s in stems):
#             valid_folders.append(song_folder)

# print("Total valid folders:", len(valid_folders))


In [ ]:
# # ========== REDEFINE ALL PREPROCESSING FUNCTIONS ==========

# import numpy as np
# import librosa
# import random
# import os

# TARGET_SR = 22050
# TARGET_DURATION = 10
# TARGET_LENGTH = TARGET_SR * TARGET_DURATION

# def load_audio(path):
#     y, sr = librosa.load(path, sr=TARGET_SR)
#     return y

# def random_crop(y):
#     if len(y) > TARGET_LENGTH:
#         start = random.randint(0, len(y) - TARGET_LENGTH)
#         y = y[start:start + TARGET_LENGTH]
#     else:
#         y = np.pad(y, (0, TARGET_LENGTH - len(y)))
#     return y

# def normalize_audio(y):
#     return y / (np.max(np.abs(y)) + 1e-6)

# def extract_log_mel(y):
#     mel = librosa.feature.melspectrogram(
#         y=y,
#         sr=TARGET_SR,
#         n_mels=128,
#         hop_length=512
#     )
#     mel_db = librosa.power_to_db(mel)
#     return mel_db

# # Correct stem names
# STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

# def create_mashup(song_folder):
#     audios = []
    
#     for stem in STEMS:
#         path = os.path.join(song_folder, stem)
        
#         if not os.path.exists(path):
#             return None
        
#         y = load_audio(path)
#         y = random_crop(y)
#         y = normalize_audio(y)
        
#         scale = random.uniform(0.6, 1.4)
#         y = y * scale
        
#         audios.append(y)
    
#     mixed = np.sum(audios, axis=0)
#     return normalize_audio(mixed)

# ESC_PATH = os.path.join(BASE_PATH, "ESC-50-master/audio")
# noise_files = os.listdir(ESC_PATH)

# def add_noise(y):
#     noise_file = random.choice(noise_files)
#     noise_path = os.path.join(ESC_PATH, noise_file)
    
#     noise = load_audio(noise_path)
#     noise = random_crop(noise)
#     noise = normalize_audio(noise)
    
#     noise_level = random.uniform(0.01, 0.05)
#     y = y + noise_level * noise
    
#     return normalize_audio(y)


In [ ]:
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader

# train_folders, val_folders = train_test_split(
#     valid_folders,
#     test_size=0.2,
#     random_state=42
# )

# train_dataset = MashupDataset(train_folders)
# val_dataset = MashupDataset(val_folders)

# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

# print("Train samples:", len(train_dataset))
# print("Val samples:", len(val_dataset))


In [ ]:
# for x, y in train_loader:
#     print("Input shape:", x.shape)
#     print("Label shape:", y.shape)
#     break


In [ ]:
# # ========== CELL 12 : CNN Architecture ==========

# import torch.nn as nn
# import torch

# class CNNModel(nn.Module):
#     def __init__(self, num_classes=10):
#         super(CNNModel, self).__init__()
        
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(64, 128, kernel_size=3, padding=1),
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#         )
        
#         # Adaptive pooling removes dimension headache
#         self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(128, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, num_classes)
#         )
    
#     def forward(self, x):
#         x = self.features(x)
#         x = self.global_pool(x)
#         x = self.classifier(x)
#         return x


# # Create model
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = CNNModel().to(device)

# print(model)


In [ ]:
# # ========== CELL 13 : Loss + Optimizer ==========

# import torch.optim as optim
# from sklearn.metrics import f1_score

# criterion = nn.CrossEntropyLoss()

# optimizer = optim.Adam(model.parameters(), lr=0.001)

# scheduler = optim.lr_scheduler.StepLR(
#     optimizer,
#     step_size=5,
#     gamma=0.5
# )

# print("setup ready")


In [ ]:
# # ========== CELL 14 : Training Loop ==========

# import torch

# EPOCHS = 10

# for epoch in range(EPOCHS):
    
#     # ----------- TRAIN -----------
#     model.train()
#     train_loss = 0
    
#     for x, y in train_loader:
#         x = x.to(device)
#         y = y.to(device)
        
#         optimizer.zero_grad()
#         outputs = model(x)
#         loss = criterion(outputs, y)
#         loss.backward()
#         optimizer.step()
        
#         train_loss += loss.item()
    
#     train_loss /= len(train_loader)
    
    
#     # ----------- VALIDATION -----------
#     model.eval()
#     val_loss = 0
#     all_preds = []
#     all_labels = []
    
#     with torch.no_grad():
#         for x, y in val_loader:
#             x = x.to(device)
#             y = y.to(device)
            
#             outputs = model(x)
#             loss = criterion(outputs, y)
#             val_loss += loss.item()
            
#             preds = torch.argmax(outputs, dim=1)
            
#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(y.cpu().numpy())
    
#     val_loss /= len(val_loader)
    
#     macro_f1 = f1_score(all_labels, all_preds, average="macro")
    
#     scheduler.step()
    
    
#     # ----------- W&B LOGGING -----------
#     wandb.log({
#         "epoch": epoch + 1,
#         "train_loss": train_loss,
#         "val_loss": val_loss,
#         "val_macro_f1": macro_f1,
#         "learning_rate": optimizer.param_groups[0]["lr"]
#     })
    
#     print(f"Epoch [{epoch+1}/{EPOCHS}] | "
#           f"Train Loss: {train_loss:.4f} | "
#           f"Val Loss: {val_loss:.4f} | "
#           f"Macro F1: {macro_f1:.4f}")
# # 

In [ ]:
import os
import random
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES_PATH = os.path.join(BASE_PATH, "genres_stems")
ESC_PATH = os.path.join(BASE_PATH, "ESC-50-master/audio")

genres = sorted(os.listdir(GENRES_PATH))
noise_files = os.listdir(ESC_PATH)

print("Setup Ready")


In [ ]:
TARGET_SR = 22050
TARGET_DURATION = 10
TARGET_LENGTH = TARGET_SR * TARGET_DURATION

STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

def load_audio(path):
    y, _ = librosa.load(path, sr=TARGET_SR)
    return y

def random_crop(y):
    if len(y) > TARGET_LENGTH:
        start = random.randint(0, len(y) - TARGET_LENGTH)
        y = y[start:start + TARGET_LENGTH]
    else:
        y = np.pad(y, (0, TARGET_LENGTH - len(y)))
    return y

def normalize(y):
    return y / (np.max(np.abs(y)) + 1e-6)


In [ ]:
def create_mashup(folder):
    genre = folder.split("/")[-2]
    genre_path = os.path.join(GENRES_PATH, genre)
    songs = os.listdir(genre_path)

    selected = random.sample(songs, 4)
    audios = []

    for i, stem in enumerate(STEMS):
        song_folder = os.path.join(genre_path, selected[i])
        path = os.path.join(song_folder, stem)

        y = load_audio(path)
        y = random_crop(y)
        y = normalize(y)

        scale = random.uniform(0.5, 1.5)
        y = y * scale
        audios.append(y)

    mixed = np.sum(audios, axis=0)
    return normalize(mixed)


def add_heavy_noise(y):
    num_noises = random.randint(1, 3)

    for _ in range(num_noises):
        noise_path = os.path.join(ESC_PATH, random.choice(noise_files))
        noise = load_audio(noise_path)
        noise = normalize(noise)

        noise = random_crop(noise)

        start = random.randint(0, TARGET_LENGTH - 1)
        end = min(start + len(noise), TARGET_LENGTH)

        y[start:end] += random.uniform(0.02, 0.08) * noise[:end-start]

    return normalize(y)


def extract_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=TARGET_SR,
        n_mels=128,
        hop_length=512
    )
    mel = librosa.power_to_db(mel)
    return mel


In [ ]:
label_map = {g: i for i, g in enumerate(genres)}

class MashupDataset(Dataset):
    def __init__(self, folders):
        self.folders = folders

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx):
        folder = self.folders[idx]
        genre = folder.split("/")[-2]
        label = label_map[genre]

        y = create_mashup(folder)
        y = add_heavy_noise(y)

        mel = extract_mel(y)

        # SpecAugment
        if random.random() > 0.5:
            f = random.randint(5, 20)
            f0 = random.randint(0, mel.shape[0] - f)
            mel[f0:f0+f, :] = 0

        if random.random() > 0.5:
            t = random.randint(10, 40)
            t0 = random.randint(0, mel.shape[1] - t)
            mel[:, t0:t0+t] = 0

        mel = torch.tensor(mel).unsqueeze(0).float()
        label = torch.tensor(label).long()

        return mel, label


In [ ]:
valid_folders = []

for g in genres:
    g_path = os.path.join(GENRES_PATH, g)
    songs = os.listdir(g_path)

    for s in songs:
        valid_folders.append(os.path.join(g_path, s))

train_f, val_f = train_test_split(valid_folders, test_size=0.2, random_state=42)

# Dataset multiply for more training samples
train_dataset = MashupDataset(train_f * 3)
val_dataset = MashupDataset(val_f)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print("Train:", len(train_dataset), "Val:", len(val_dataset))


In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = CNNModel().to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 15

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    preds_all = []
    labels_all = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item()

            preds = torch.argmax(out, 1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(y.cpu().numpy())

    val_loss /= len(val_loader)
    macro_f1 = f1_score(labels_all, preds_all, average="macro")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": macro_f1
    })

    print(f"Epoch {epoch+1} | F1: {macro_f1:.4f}")


In [ ]:
model.eval()
print("Model set to evaluation mode")


In [1]:
!pip uninstall -y protobuf -q
!pip install protobuf==3.20.3 -q
!pip install transformers==4.37.2 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
a2a-sdk 0.3.22 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12

In [2]:
# =========================
# AST CLEAN START - CELL 1
# =========================

import os
import numpy as np
import random
import librosa
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import ASTFeatureExtractor, ASTForAudioClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cuda


In [3]:
# =========================
# AST CLEAN START - CELL 2
# =========================

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES_PATH = os.path.join(BASE_PATH, "genres_stems")

# Genres list
genres = sorted(os.listdir(GENRES_PATH))
label_map = {genre: idx for idx, genre in enumerate(genres)}

# Collect all song folders
all_folders = []

for genre in genres:
    genre_path = os.path.join(GENRES_PATH, genre)
    for folder in os.listdir(genre_path):
        folder_path = os.path.join(genre_path, folder)
        if os.path.isdir(folder_path):
            all_folders.append(folder_path)

# Train / Validation split
train_f, val_f = train_test_split(
    all_folders,
    test_size=0.2,
    random_state=42,
    stratify=[f.split("/")[-2] for f in all_folders]
)

print("Total:", len(all_folders))
print("Train:", len(train_f))
print("Val:", len(val_f))


Total: 1000
Train: 800
Val: 200


In [4]:
# =========================
# AST CLEAN START - CELL 3
# =========================

feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

model_ast = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

model_ast.to(device)

# IMPORTANT: Unfreeze everything (full fine-tuning)
for param in model_ast.parameters():
    param.requires_grad = True

print("AST Model Loaded & Fully Unfrozen")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AST Model Loaded & Fully Unfrozen


In [36]:
# =========================
# AST CLEAN START - CELL 4 (STABLE VERSION)
# =========================

AST_SR = 16000
AST_DURATION = 10
AST_LENGTH = AST_SR * AST_DURATION

STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

def create_strong_mashup(folder):

    audios = []

    selected = STEMS  # always use all stems

    for stem in selected:
        path = os.path.join(folder, stem)

        if not os.path.exists(path):
            continue

        y, _ = librosa.load(path, sr=AST_SR)

        # random crop
        if len(y) > AST_LENGTH:
            start = random.randint(0, len(y) - AST_LENGTH)
            y = y[start:start + AST_LENGTH]
        else:
            y = np.pad(y, (0, AST_LENGTH - len(y)))

        
        if random.random() < 0.3:
            rate = random.uniform(0.95, 1.05)
            y = librosa.effects.time_stretch(y=y, rate=rate)

            if len(y) > AST_LENGTH:
                y = y[:AST_LENGTH]
            else:
                y = np.pad(y, (0, AST_LENGTH - len(y)))

      

        # gain scaling (light)
        y = y * random.uniform(0.8, 1.2)

        audios.append(y)

    if len(audios) == 0:
        mixed = np.zeros(AST_LENGTH)
    else:
        mixed = np.sum(audios, axis=0)

    # 🔥 LIGHT noise (reduced)
    noise_level = random.uniform(0.01, 0.03)
    noise = np.random.randn(AST_LENGTH)
    mixed = mixed + noise_level * noise

    mixed = mixed / (np.max(np.abs(mixed)) + 1e-6)

    return mixed


In [37]:
# =========================
# AST CLEAN START - CELL 5
# =========================

class StrongASTDataset(Dataset):
    def __init__(self, folders):
        self.folders = folders

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx):

        folder = self.folders[idx]
        genre = folder.split("/")[-2]
        label = label_map[genre]

        # 🔥 strong augmentation
        y = create_strong_mashup(folder)

        inputs = feature_extractor(
            y,
            sampling_rate=AST_SR,
            return_tensors="pt"
        )

        input_values = inputs["input_values"].squeeze(0)

        return input_values, torch.tensor(label)


In [38]:
# =========================
# AST CLEAN START - CELL 6
# =========================

train_dataset = StrongASTDataset(train_f)
val_dataset = StrongASTDataset(val_f)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,   # safe for AST
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoader Ready")


DataLoader Ready


In [39]:
# =========================
# AST CLEAN START - CELL 7
# =========================

from transformers import get_cosine_schedule_with_warmup
from torch.cuda.amp import GradScaler, autocast

EPOCHS = 20

# Label smoothing helps generalization
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model_ast.parameters(),
    lr = 6e-6,         # slightly smaller for stability
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

scaler = GradScaler()

print("Optimizer + Scheduler + AMP Ready")


Optimizer + Scheduler + AMP Ready


/tmp/ipykernel_55/973597412.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [40]:
# =========================
# AST CLEAN START - CELL 8
# =========================

best_f1 = 0

for epoch in range(EPOCHS):

    # -------- TRAIN --------
    model_ast.train()
    total_loss = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model_ast(x).logits
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()

        # gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_ast.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # -------- VALIDATION --------
    model_ast.eval()
    preds_all = []
    labels_all = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            outputs = model_ast(x).logits
            preds = torch.argmax(outputs, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(y.cpu().numpy())

    macro_f1 = f1_score(labels_all, preds_all, average="macro")

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | F1: {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model_ast.state_dict(), "best_ast_model_strong.pth")
        print("New Best Model Saved!")

print("\nTraining Complete")
print("Best F1:", best_f1)


/tmp/ipykernel_55/2242215286.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/20 | Loss: 0.6179 | F1: 0.8303
New Best Model Saved!


/tmp/ipykernel_55/2242215286.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2/20 | Loss: 0.6342 | F1: 0.7824


/tmp/ipykernel_55/2242215286.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch 18/20 | Loss: 0.5213 | F1: 0.8371


/tmp/ipykernel_55/2242215286.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 19/20 | Loss: 0.5288 | F1: 0.8227


/tmp/ipykernel_55/2242215286.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 20/20 | Loss: 0.5256 | F1: 0.8568

Training Complete
Best F1: 0.871873769519581


In [41]:
model_ast.load_state_dict(torch.load("best_ast_model_strong.pth"))
model_ast.eval()

print("Strong AST model loaded")


Strong AST model loaded


In [42]:
def preprocess_test_tta(path, num_crops=7):

    y, _ = librosa.load(path, sr=AST_SR)

    crops = []

    for _ in range(num_crops):

        if len(y) > AST_LENGTH:
            start = random.randint(0, len(y) - AST_LENGTH)
            crop = y[start:start + AST_LENGTH]
        else:
            crop = np.pad(y, (0, AST_LENGTH - len(y)))

        inputs = feature_extractor(
            crop,
            sampling_rate=AST_SR,
            return_tensors="pt"
        )

        crops.append(inputs["input_values"].squeeze(0))

    return crops


In [43]:
import pandas as pd

test_df = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))

predictions = []

model_ast.eval()

with torch.no_grad():

    for idx in range(len(test_df)):

        filename = test_df.iloc[idx]["filename"]
        audio_path = os.path.join(BASE_PATH, filename)

        crops = preprocess_test_tta(audio_path, num_crops=5)

        logits_sum = 0

        for crop in crops:
            crop = crop.unsqueeze(0).to(device)
            outputs = model_ast(crop).logits
            logits_sum += outputs

        avg_logits = logits_sum / len(crops)

        pred = torch.argmax(avg_logits, dim=1).item()

        predictions.append(genres[pred])

        if idx % 200 == 0:
            print(f"Processed {idx}/{len(test_df)}")

print("TTA Prediction Done ")


Processed 0/3020
Processed 200/3020
Processed 400/3020
Processed 600/3020
Processed 800/3020
Processed 1000/3020
Processed 1200/3020
Processed 1400/3020
Processed 1600/3020
Processed 1800/3020
Processed 2000/3020
Processed 2200/3020
Processed 2400/3020
Processed 2600/3020
Processed 2800/3020
Processed 3000/3020
TTA Prediction Done 


In [45]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission.csv", index=False)

print("Submission file created")
submission.head()


Submission file created


,id,genre
0,1,pop
1,2,jazz
2,3,disco
3,4,metal
4,5,country


In [ ]:
# =========================
# AST SUBMISSION - CELL 2
# =========================

def preprocess_test_audio_ast(path):
    
    y, _ = librosa.load(path, sr=AST_SR)

    # fix length to 10 sec
    if len(y) > AST_LENGTH:
        y = y[:AST_LENGTH]
    else:
        y = np.pad(y, (0, AST_LENGTH - len(y)))

    inputs = feature_extractor(
        y,
        sampling_rate=AST_SR,
        return_tensors="pt"
    )

    input_values = inputs["input_values"].squeeze(0)

    return input_values


In [ ]:
# =========================
# AST SUBMISSION - CELL 3
# =========================

import pandas as pd

test_df = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))

print("Total test samples:", len(test_df))

predictions = []

model_ast.eval()

with torch.no_grad():
    for idx in range(len(test_df)):
        
        file_id = test_df.iloc[idx]["id"]
        filename = test_df.iloc[idx]["filename"]
        
        # 🔥 CORRECT PATH (important)
        audio_path = os.path.join(BASE_PATH, filename)
        
        x = preprocess_test_audio_ast(audio_path).to(device)
        
        outputs = model_ast(x.unsqueeze(0)).logits
        pred = torch.argmax(outputs, dim=1).item()
        
        predictions.append(genres[pred])
        
        if idx % 200 == 0:
            print(f"Processed {idx}/{len(test_df)}")

print("Prediction Done ")


In [ ]:
# =========================
# AST SUBMISSION - CELL 4
# =========================

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission.csv", index=False)

print("Submission file created ")
submission.head()


In [1]:
# =========================
# INSTALL DEPENDENCIES
# =========================

!pip install -q transformers
!pip install -q datasets
!pip install -q librosa
!pip install -q torchmetrics
!pip install -q wandb
!pip install -q audiomentations


# =========================
# IMPORT LIBRARIES
# =========================

import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import librosa
import soundfile as sf

from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import f1_score
from torchmetrics.classification import MulticlassF1Score

import warnings
warnings.filterwarnings("ignore")

print("All libraries loaded successfully")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 11.3 MB/s eta 0:00:00


2026-02-17 17:12:24.806839: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771348345.014081      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771348345.068423      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771348345.533979      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771348345.534034      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771348345.534038      55 computation_placer.cc:177] computation placer alr

All libraries loaded successfully


In [3]:
# =========================
# DATA PATHS
# =========================

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

GENRES_PATH = os.path.join(BASE_PATH, "genres_stems")
MASHUP_PATH = os.path.join(BASE_PATH, "mashups")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")

# =========================
# LABEL SETUP
# =========================

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

# =========================
# CONFIG
# =========================

SAMPLE_RATE = 16000
MAX_DURATION = 12
MAX_LENGTH = SAMPLE_RATE * MAX_DURATION

BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)


Device: cuda


In [4]:
import librosa
import numpy as np
import random
import os

# =========================
# AUDIO HELPERS
# =========================

def load_audio(path, sr=SAMPLE_RATE):
    audio, _ = librosa.load(path, sr=sr)
    return audio


def crop_or_pad(audio):
    if len(audio) > MAX_LENGTH:
        start = random.randint(0, len(audio) - MAX_LENGTH)
        audio = audio[start:start + MAX_LENGTH]
    else:
        pad = MAX_LENGTH - len(audio)
        audio = np.pad(audio, (0, pad))
    return audio


def tempo_augment(audio):
    rate = random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(audio, rate=rate)


# =========================
# LOAD ESC-50 NOISE FILES
# =========================

ESC_PATH = os.path.join(BASE_PATH, "ESC-50-master", "audio")

noise_files = []
for root, _, files in os.walk(ESC_PATH):
    for f in files:
        if f.endswith(".wav"):
            noise_files.append(os.path.join(root, f))

print("Noise files:", len(noise_files))


# =========================
# STRONG NOISE VERSION
# =========================

def add_noise(audio):

    # first noise
    noise_path = random.choice(noise_files)
    noise = load_audio(noise_path)
    noise = crop_or_pad(noise)

    noise_level = random.uniform(0.05, 0.2)

    audio = audio + noise_level * noise

    # sometimes add second noise
    if random.random() > 0.7:
        noise_path2 = random.choice(noise_files)
        noise2 = load_audio(noise_path2)
        noise2 = crop_or_pad(noise2)
        audio = audio + random.uniform(0.05, 0.2) * noise2

    return audio


# =========================
# BUILD SAFE SONG INDEX
# =========================

REQUIRED_STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

def build_song_index():
    index = {}

    for genre in GENRES:
        genre_path = os.path.join(GENRES_PATH, genre)
        valid_songs = []

        for folder in os.listdir(genre_path):
            folder_path = os.path.join(genre_path, folder)

            if not os.path.isdir(folder_path):
                continue

            if all(os.path.exists(os.path.join(folder_path, stem)) for stem in REQUIRED_STEMS):
                valid_songs.append(folder_path)

        index[genre] = valid_songs
        print(genre, ":", len(valid_songs))

    return index


song_index = build_song_index()


Noise files: 2000
blues : 100
classical : 100
country : 100
disco : 100
hiphop : 100
jazz : 100
metal : 100
pop : 100
reggae : 100
rock : 100


In [5]:
from torch.utils.data import Dataset

class SyntheticMashupDataset(Dataset):
    def __init__(self, song_index, feature_extractor):
        self.song_index = song_index
        self.feature_extractor = feature_extractor

    def __len__(self):
        return 5000  # larger synthetic dataset

    def mix_stems(self, stem_paths):
        mixed = None

        for path in stem_paths:
            audio = load_audio(path)
            audio = tempo_augment(audio)
            audio = crop_or_pad(audio)

            # random gain scaling (important upgrade)
            gain = random.uniform(0.6, 1.4)
            audio = audio * gain

            if mixed is None:
                mixed = audio
            else:
                mixed += audio

        mixed = mixed / (np.max(np.abs(mixed)) + 1e-6)
        return mixed

    def __getitem__(self, idx):

        while True:
            genre = random.choice(GENRES)
            songs = self.song_index[genre]

            try:
                # 50% synthetic cross-song mix
                if random.random() > 0.7:
                    drums_song = random.choice(songs)
                    vocals_song = random.choice(songs)
                    bass_song = random.choice(songs)
                    other_song = random.choice(songs)
                else:
                    # 50% clean same-song mix (reduces domain gap)
                    same_song = random.choice(songs)
                    drums_song = vocals_song = bass_song = other_song = same_song

                stem_paths = [
                    os.path.join(drums_song, "drums.wav"),
                    os.path.join(vocals_song, "vocals.wav"),
                    os.path.join(bass_song, "bass.wav"),
                    os.path.join(other_song, "other.wav"),
                ]

                audio = self.mix_stems(stem_paths)
                audio = add_noise(audio)

                inputs = self.feature_extractor(
                    audio,
                    sampling_rate=SAMPLE_RATE,
                    return_tensors="pt"
                )

                label = label2id[genre]

                return {
                    "input_values": inputs["input_values"].squeeze(0),
                    "labels": torch.tensor(label)
                }

            except Exception:
                continue


In [7]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=NUM_LABELS,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

model.to(DEVICE)

print("Model and feature extractor ready")


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model and feature extractor ready


In [8]:
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassF1Score

# =========================
# CREATE DATASET
# =========================

train_dataset = SyntheticMashupDataset(
    song_index=song_index,
    feature_extractor=feature_extractor
)

print("Synthetic dataset ready")


# =========================
# DATALOADER
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("DataLoader ready")


# =========================
# MACRO F1 METRIC
# =========================

f1_metric = MulticlassF1Score(
    num_classes=NUM_LABELS,
    average="macro"
).to(DEVICE)

print("Macro F1 metric ready")


Synthetic dataset ready
DataLoader ready
Macro F1 metric ready


In [9]:
# =========================
# FREEZE BACKBONE (First Phase)
# =========================

for param in model.base_model.parameters():
    param.requires_grad = False

print("Backbone frozen")



import torch.nn as nn
from tqdm import tqdm

# =========================
# OPTIMIZER + LOSS
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()

print("Optimizer ready")


# =========================
# TRAINING LOOP
# =========================

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0
    f1_metric.reset()

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        input_values = batch["input_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_values=input_values)
        logits = outputs.logits

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        f1_metric.update(preds, labels)

        progress_bar.set_description(f"Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    macro_f1 = f1_metric.compute().item()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")


Backbone frozen
Optimizer ready


Loss: 1.7443: 100%|██████████| 313/313 [36:44<00:00,  7.04s/it]



Epoch 1/5
Average Loss: 2.1543
Macro F1: 0.2230


Loss: 1.2019: 100%|██████████| 313/313 [33:26<00:00,  6.41s/it]



Epoch 2/5
Average Loss: 1.6348
Macro F1: 0.5222


Loss: 1.0163: 100%|██████████| 313/313 [33:02<00:00,  6.33s/it]



Epoch 3/5
Average Loss: 1.3466
Macro F1: 0.6235


Loss: 0.9648:  58%|█████▊    | 181/313 [19:17<14:04,  6.40s/it]


KeyboardInterrupt: 

In [23]:
import pandas as pd
from tqdm import tqdm

TTA_STEPS = 5

test_df = pd.read_csv(TEST_CSV)
print("Test samples:", len(test_df))

model.eval()
predictions = []

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

        # correct file path
        if "filename" in test_df.columns:
            file_path = os.path.join(BASE_PATH, row["filename"])
        else:
            file_path = os.path.join(MASHUP_PATH, f"{row['id']}.wav")

        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)

        logits_sum = torch.zeros(NUM_LABELS).to(DEVICE)

        for _ in range(TTA_STEPS):

            if len(audio) > MAX_LENGTH:
                start = random.randint(0, len(audio) - MAX_LENGTH)
                audio_crop = audio[start:start + MAX_LENGTH]
            else:
                pad = MAX_LENGTH - len(audio)
                audio_crop = np.pad(audio, (0, pad))

            inputs = feature_extractor(
                audio_crop,
                sampling_rate=SAMPLE_RATE,
                return_tensors="pt"
            )

            input_values = inputs["input_values"].to(DEVICE)

            outputs = model(input_values=input_values)
            logits_sum += outputs.logits.squeeze(0)

        avg_logits = logits_sum / TTA_STEPS
        pred_id = torch.argmax(avg_logits).item()
        pred_label = id2label[pred_id]

        predictions.append(pred_label)


submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission_5tta.csv", index=False)

print("submission_5tta.csv saved")


Test samples: 3020


100%|██████████| 3020/3020 [22:51<00:00,  2.20it/s]

submission_5tta.csv saved
